# Ragionare e agire: il ciclo dell'agente

Il codice del capitolo [«Ragionare e agire: il ciclo dell'agente»](https://book.paithon.it/main/Agenti/agenti-e-tool-use.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Ragionare e agire: il ciclo dell'agente

[Leggi la pagina](https://book.paithon.it/main/Agenti/agenti-e-tool-use.html)


### Un agente giocattolo, in Python


In [ ]:
import ast
import operator

# --- due strumenti reali ---

# operatori ammessi: un mini-interprete sicuro, niente eval()
_OP = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.USub: operator.neg,
}

def _operatore(op):
    """Un operatore fuori elenco esce di qui con un errore leggibile."""
    if type(op) not in _OP:
        raise ValueError(f"operatore non ammesso: {type(op).__name__}")
    return _OP[type(op)]

def _valuta(nodo):
    if isinstance(nodo, ast.Constant):        # un numero, e solo un numero
        numero = isinstance(nodo.value, (int, float))
        if not numero or isinstance(nodo.value, bool):   # in Python True e' 1
            raise ValueError("ammessi solo numeri")
        return nodo.value
    if isinstance(nodo, ast.BinOp):           # a operatore b
        return _operatore(nodo.op)(_valuta(nodo.left), _valuta(nodo.right))
    if isinstance(nodo, ast.UnaryOp):         # -a
        return _operatore(nodo.op)(_valuta(nodo.operand))
    raise ValueError("espressione non ammessa")

def calcola(espressione):
    """Valuta un'espressione aritmetica in modo sicuro (senza eval)."""
    return _valuta(ast.parse(espressione, mode="eval").body)

# un piccolo archivio: la memoria esterna che il modello non ha nei pesi
ARCHIVIO = {
    "attention is all you need": "2017",
    "gpt-3": "2020",
    "react": "2022",
}

def cerca(chiave):
    """Cerca un fatto nell'archivio; restituisce sempre una stringa."""
    return ARCHIVIO.get(chiave.lower().strip(), "non trovato")

STRUMENTI = {"calcola": calcola, "cerca": cerca}

In [ ]:
# --- l'LLM finto: deterministico, a regole ---

def llm_finto(traccia):
    """Data la traccia finora, emette (pensiero, azione, argomento).
    Un vero LLM genererebbe questo testo; qui lo decide una regola."""
    ultima = traccia[-1]["osservazione"] if traccia else None
    if ultima is None:
        return ("Non conosco a memoria l'anno del paper: lo cerco.",
                "cerca", "attention is all you need")
    if ultima == "non trovato":     # la ricerca a vuoto: non si inventa
        return ("L'archivio non ha quella voce, e a memoria non la so.",
                "Answer", "non lo so")
    if ultima == "2017":
        return ("Il paper è del 2017. Calcolo quanti anni fa, dal 2026.",
                "calcola", "2026 - 2017")
    return (f"Il calcolo dice {ultima}: ho tutto per rispondere.",
            "Answer", "'Attention Is All You Need' è del 2017: 9 anni fa nel 2026.")

# --- il ciclo dell'agente ---

def esegui_agente(domanda, max_passi=5):
    print(f"Domanda: {domanda}\n")
    traccia = []
    for _ in range(max_passi):
        pensiero, azione, argomento = llm_finto(traccia)   # il modello "ragiona"
        print(f"Thought: {pensiero}")
        if azione == "Answer":                             # fine del loop
            print(f"Answer: {argomento}")
            return argomento
        print(f"Action: {azione}[{argomento}]")
        osservazione = str(STRUMENTI[azione](argomento))   # il sistema agisce
        print(f"Observation: {osservazione}\n")
        traccia.append({"azione": azione, "argomento": argomento,
                        "osservazione": osservazione})      # torna nel contesto
    print("(limite di passi raggiunto)")

esegui_agente("In che anno è uscito 'Attention Is All You Need' "
              "e quanti anni fa è, nel 2026?")

## RAG avanzato: oltre il recupero ingenuo

[Leggi la pagina](https://book.paithon.it/main/Agenti/rag-avanzato.html)


### Riordinare i candidati: il reranking


In [ ]:
import torch
from itertools import combinations

# --- Stadio 1: recupero grezzo con il bi-encoder ---
# stesso mini-archivio della sezione «Cercare per rispondere»:
# quattro dimensioni leggibili [gatti, muri/casa, automobili, cucina].
passaggi = [
    "Il gatto nero salta sul muro del giardino.",
    "Il muro portante sostiene il solaio.",
    "La vettura elettrica si ricarica in garage.",
    "L'auto storica sfila per il centro.",
    "Il gatto dorme accanto ai fornelli.",
    "La ricetta prevede burro e salvia.",
]
E = torch.tensor([
    [0.9, 0.6, 0.0, 0.1],
    [0.1, 0.9, 0.1, 0.0],
    [0.0, 0.1, 0.9, 0.0],
    [0.1, 0.0, 0.8, 0.1],
    [0.8, 0.2, 0.0, 0.5],
    [0.0, 0.1, 0.1, 0.9],
])
E = E / E.norm(dim=1, keepdim=True)   # righe normalizzate: prodotto = coseno

domanda = "Su cosa salta il gatto nero?"
q = torch.tensor([0.9, 0.7, 0.0, 0.0])
q = q / q.norm()

# il bi-encoder e' veloce, quindi copre tutto l'archivio (qui sei passaggi, e
# li confrontiamo davvero uno a uno; in un archivio vero si userebbero gli
# strati della figura, che coprono tutto senza toccare tutto). Ma e' grezzo.
# recuperiamo di proposito piu' candidati di quanti ne serviranno: sono il
# terreno di caccia dello stadio successivo.
sim = E @ q
val, cand = torch.topk(sim, k=4)

print("Stadio 1 - bi-encoder (veloce, tutto l'archivio):")
for v, i in zip(val, cand):
    print(f"  coseno {v:.2f}  {passaggi[i]}")

# --- Stadio 2: reranking con un "cross-encoder" didattico ---
# Il bi-encoder ha collassato ogni frase in un unico vettore: cosi' un
# passaggio che condivide solo il tema "gatto" gli sembra vicino. Un
# cross-encoder legge domanda e passaggio INSIEME. Qui lo simuliamo con i
# concetti dei due testi, premiando le CO-OCCORRENZE: a rispondere non e' il
# gatto, ne' il saltare, ma un gatto CHE SALTA.
concetti = {
    domanda: {"gatto", "nero", "saltare"},
    0: {"gatto", "nero", "saltare", "muro", "giardino"},  # copre tutto: risponde
    1: {"muro", "portante", "solaio"},
    2: {"vettura", "garage"},
    3: {"auto", "centro"},
    4: {"gatto", "dormire", "fornelli"},   # solo il tema "gatto": quasi-pertinente
    5: {"ricetta", "burro"},
}

def cross_encoder(domanda, i):
    """Punteggio della COPPIA (domanda, passaggio), non del solo passaggio.
    Ogni concetto della domanda coperto vale 1; ogni coppia di concetti
    coperti INSIEME vale 2, perche' e' la combinazione a rispondere."""
    coperti = concetti[domanda] & concetti[i]
    return len(coperti) + 2 * len(list(combinations(coperti, 2)))

# il cross-encoder e' costoso: lo applichiamo SOLO ai candidati dello stadio 1
riordino = sorted(cand.tolist(), key=lambda i: cross_encoder(domanda, i),
                  reverse=True)

print("\nStadio 2 - cross-encoder (preciso, solo sui 4 candidati):")
for i in riordino:
    print(f"  pertinenza {cross_encoder(domanda, i):.1f}  {passaggi[i]}")

# ora che i punteggi sono separati, una soglia ha senso: passa solo chi si
# avvicina al migliore, invece di riempire a forza un numero fisso di posti.
# Il minimo assoluto non e' un dettaglio: se nessuno rispondesse varrebbero
# tutti zero, e meta' di zero lascia passare tutti.
migliore = cross_encoder(domanda, riordino[0])
soglia = max(migliore / 2, 1)
rosa = [i for i in riordino if cross_encoder(domanda, i) >= soglia]

print("\nAl generatore va solo chi supera meta' del punteggio migliore:")
for n, i in enumerate(rosa, 1):
    print(f"  [{n}] {passaggi[i]}")

## Context engineering: il contesto è l'interfaccia

[Leggi la pagina](https://book.paithon.it/main/Agenti/context-engineering.html)


### Assemblare il contesto, con un budget


In [ ]:
# Un "context builder": dato un budget di token, assembla il prompt
# scegliendo i passaggi piu' importanti, troncando o scartando il resto,
# e collocando il pezzo piu' rilevante IN FONDO (contro il "lost in the middle").

def conta_token(testo):
    """Stima i token contando le parole: grezza, ma sufficiente per il budget."""
    return len(testo.split())

# System prompt e domanda sono obbligatori: entrano sempre, non si toccano.
system_prompt = (
    "Sei un assistente che risponde citando solo i passaggi forniti. "
    "Se l'informazione non c'e', dillo."
)
domanda = "In che anno e' stato pubblicato il paper sui Transformer?"

# I passaggi recuperati, ciascuno con una rilevanza (piu' alta = piu' utile).
passaggi = [
    (0.95, "Il paper 'Attention Is All You Need' introduce i Transformer nel 2017."),
    (0.20, "Le reti convoluzionali dominarono la visione artificiale negli anni 2010."),
    (0.60, "L'architettura Transformer abbandona la ricorrenza in favore dell'attenzione."),
    (0.10, "Il primo modello GPT fu addestrato su un corpus di libri."),
    (0.75, "L'attenzione scaled dot-product e' il cuore del Transformer."),
]


def riga_fonte(punteggio, testo, troncato=False):
    """La riga come finira' nel prompt. Il marcatore e' testo anche lui:
    entra nella finestra, quindi si paga e va contato."""
    return f"[fonte {punteggio:.2f}{' (troncata)' if troncato else ''}] {testo}"


COSTO_MARCATORE = conta_token(riga_fonte(0.0, "", troncato=True))


def costruisci_contesto(system_prompt, passaggi, domanda, budget):
    """Assembla un prompt che sta nel budget di token.
    Obbligatori: system prompt e domanda. I passaggi entrano per rilevanza
    decrescente finche' c'e' spazio; l'ultimo che sfora viene troncato; la
    disposizione finale e' a V, il migliore in fondo e il secondo in testa."""
    coda = f"Domanda: {domanda}"
    residuo = budget - conta_token(system_prompt) - conta_token(coda)
    if residuo < 0:
        raise ValueError("budget insufficiente perfino per system prompt e domanda")

    ordinati = sorted(passaggi, key=lambda p: p[0], reverse=True)
    scelti = []  # (punteggio, testo, troncato?), gia' per rilevanza decrescente
    for punteggio, testo in ordinati:
        costo = conta_token(riga_fonte(punteggio, testo))
        if costo <= residuo:                          # ci sta intero
            scelti.append((punteggio, testo, False))
            residuo -= costo
        elif residuo >= COSTO_MARCATORE + 2:          # non ci sta: lo tronco
            quante = residuo - COSTO_MARCATORE - 1    # -1 per il segno di taglio
            troncato = " ".join(testo.split()[:quante]) + " …"
            scelti.append((punteggio, troncato, True))
            residuo -= conta_token(riga_fonte(punteggio, troncato, True))
            break
        # altrimenti lo scarto e provo il prossimo (piu' corto o meno rilevante)

    # "lost in the middle": la curva e' a U, si legge bene all'inizio E alla
    # fine. Disposizione a V: il migliore in coda (a ridosso della domanda),
    # il secondo in testa, i peggiori sepolti nel mezzo.
    testa, fondo = [], []
    for n, scelto in enumerate(scelti):
        (fondo if n % 2 == 0 else testa).append(scelto)
    corpo = "\n".join(riga_fonte(p, txt, t) for p, txt, t in testa + fondo[::-1])

    prompt = f"{system_prompt}\n\n{corpo}\n\n{coda}"
    return prompt, conta_token(prompt)   # il conto vero, marcatori compresi


BUDGET = 58
prompt, usati = costruisci_contesto(system_prompt, passaggi, domanda, BUDGET)
print(prompt)
print(f"\nToken usati: {usati}/{BUDGET}")

# quanto pesa il montaggio: anche i marcatori [fonte 0.95] sono testo
righe = [r for r in prompt.split("\n") if r.startswith("[fonte")]
pezzi = (conta_token(system_prompt) + conta_token(f"Domanda: {domanda}")
         + sum(conta_token(r.split("] ", 1)[1]) for r in righe))
print(f"i soli pezzi scelti: {pezzi} token; i marcatori: {usati - pezzi}, "
      f"cioe' il {(usati - pezzi) / pezzi:.0%} in piu'")

## Architetture di agenti e come valutarli

[Leggi la pagina](https://book.paithon.it/main/Agenti/architetture-e-valutazione.html)


### Valutare un agente: il problema difficile


In [ ]:
import math

# ogni episodio: esito, passi, token consumati, traiettoria valida?
episodi = [
    {"successo": True,  "passi": 4,  "token": 2100, "traiettoria_ok": True},
    {"successo": True,  "passi": 9,  "token": 5400, "traiettoria_ok": False},
    {"successo": False, "passi": 12, "token": 8000, "traiettoria_ok": False},
    {"successo": True,  "passi": 5,  "token": 2600, "traiettoria_ok": True},
    {"successo": False, "passi": 6,  "token": 3100, "traiettoria_ok": True},
]

n = len(episodi)
successi = [e for e in episodi if e["successo"]]
tasso_successo = len(successi) / n
# fra i compiti riusciti, quanti per una strada "pulita"?
traiettorie_ok = sum(e["traiettoria_ok"] for e in successi) / len(successi)
token_medi = sum(e["token"] for e in episodi) / n

print(f"episodi: {n}")
print(f"tasso di successo: {tasso_successo:.0%}")
print(f"successi con traiettoria valida: {traiettorie_ok:.0%}")
print(f"token medi per episodio: {token_medi:.0f}")

# quanto vale davvero quel 60%? Fra quali due valori puo' stare il vero tasso
# di successo, viste cosi' poche prove? La formula qui sotto e' l'intervallo di
# Wilson, che regge anche su pochi episodi (la formula ingenua, con pochi dati,
# darebbe estremi sotto zero o sopra il cento per cento).
# z = 1.96 e' il numero che corrisponde al "95 per cento di fiducia": lo si
# legge sulle tavole della distribuzione normale e non si ricava a mano.
z = 1.96
p_succ = tasso_successo          # attenzione: qui e' il tasso di SUCCESSO,
                                 # non la probabilita' di sbagliare un passo
centro = (p_succ + z**2 / (2*n)) / (1 + z**2 / n)
raggio = z * math.sqrt(p_succ*(1-p_succ)/n + z**2 / (4*n**2)) / (1 + z**2 / n)
print(f"intervallo al 95%: da {centro - raggio:.0%} a {centro + raggio:.0%}")

# e quanti episodi servirebbero perche' l'incertezza scenda a 5 punti
# percentuali? Qui torna comoda la formula ingenua, che per dimensionare un
# esperimento basta: n = z^2 * p_succ * (1-p_succ) / errore^2, cioe'
# 3.84 * 0.24 / 0.0025.
print(f"episodi per +/- 5 punti: {z**2 * p_succ * (1-p_succ) / 0.05**2:.0f}")